### GBIF Data Transformation: Great-tailed Grackle (*Quiscalus mexicanus*)
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno  
**Source:** [GBIF API](https://www.gbif.org/)  
**Goal:** Prototype Bronze → Silver transformations including type casting, schema standardization, deduplication, and dropping sparse fields before implementing in the production Spark job.  

#### Setup Spark Session and Load Raw GCS Data

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when 
from dotenv import load_dotenv
from pathlib import Path 

In [ ]:
load_dotenv()

In [ ]:
GCS_BUCKET = os.getenv('GCS_BUCKET')
BRONZE_PATH = f'gs://{GCS_BUCKET}/bronze_gbif/gbif_occurrences'
JAR_PATH = str(Path().resolve().parents[1] / 'jars' / 'gcs-connector.jar')

In [ ]:
# initialize Spark session with GCS connector
spark = SparkSession.builder \
    .appName('gbif_bronze_to_silver') \
    .config('spark.jars', JAR_PATH) \
    .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem') \
    .config('spark.hadoop.fs.AbstractFileSystem.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS') \
    .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true') \
    .config('spark.hadoop.google.cloud.auth.service.account.json.keyfile', os.getenv('GOOGLE_APPLICATION_CREDENTIALS')) \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark initialized with local GCS connector.")

In [ ]:
# read raw parquet from GCS bronze
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f'Rows: {bronze_df.count():,}')
print(f'Columns: {len(bronze_df.columns)}')

#### Schema Enforcement & Column Pruning
Selecting 22 columns from 151 raw fields. Drops dlt internals, flattened classification UUIDs, and sparse metadata fields identified during exploration.

In [ ]:
SILVER_COLUMNS = [
    # identifiers
    'gbif_id',
    'occurrence_id',
    
    # taxonomy
    'scientific_name',
    'species',
    'iucn_red_list_category',
    
    # spatial 
    'decimal_latitude',
    'decimal_longitude',
    'coordinate_uncertainty_in_meters',
    'state_province',
    'county',
    'locality',
    
    # temporal
    'event_date',
    'year',
    'month',
    'day',
    
    # biological/behavioral 
    'individual_count',
    'occurrence_status',
    'basis_of_record',
    'sex',
    
    # audit & context
    'dataset_name',
    'recorded_by',
    'license'
]

In [ ]:
silver_df = bronze_df.select(SILVER_COLUMNS)
print(f'Columns: {len(silver_df.columns)}')
silver_df.printSchema()

#### Deduplication
Ensures one record per occurrence across pipeline runs. Critical at scale since GBIF aggregates from overlapping providers, which directly impacts BigQuery storage and dbt processing costs.

In [ ]:
initial_count = silver_df.count()
silver_df = silver_df.dropDuplicates(['gbif_id'])
current_count = silver_df.count()
print(f'Removed {initial_count - current_count:,} duplicate records.')
print(f'Remaining records: {current_count:,}')

#### Filter Invalid Records
Dropping records with missing coordinates or event date. Both are required for spatial and temporal analysis downstream.

In [ ]:
# drop records with missing coordinates or event date
silver_df = silver_df.dropna(subset=['decimal_latitude', 
                                     'decimal_longitude', 
                                     'event_date'])
print(f'Remaining records: {silver_df.count()}')

#### Type Casting & Column Renaming
Casting numeric fields from long to integer, converting event_date to date type, and standardizing verbose column names for cleaner downstream usage in dbt and BigQuery.

In [ ]:
# type casting
silver_df = silver_df \
    .withColumn('year', f.col('year').cast('integer')) \
    .withColumn('month', f.col('month').cast('integer')) \
    .withColumn('day', f.col('day').cast('integer')) \
    .withColumn('individual_count', f.col('individual_count').cast('integer')) \
    .withColumn('event_date', f.col('event_date').cast('date'))

In [ ]:
# rename columns
silver_df = silver_df \
    .withColumnRenamed('decimal_latitude', 'latitude') \
    .withColumnRenamed('decimal_longitude', 'longitude') \
    .withColumnRenamed('coordinate_uncertainty_in_meters', 'coordinate_uncertainty') \
    .withColumnRenamed('iucn_red_list_category', 'iucn_category') \
    .withColumnRenamed('occurrence_status', 'status') \
    .withColumnRenamed('basis_of_record', 'record_type') \
    .withColumnRenamed('individual_count', 'count') \
    .withColumnRenamed('recorded_by', 'observer') \
    .withColumnRenamed('dataset_name', 'source_dataset') \
    .withColumnRenamed('state_province', 'state')

In [ ]:
silver_df.printSchema()

#### Null Handling
Auditing null counts to inform handling decisions, then applying fixes before writing to Silver.

In [ ]:
# null audit before handling
null_counts = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

In [ ]:
# drop columns too sparse to be useful
# county: ~99% null, locality: ~93% null
silver_df = silver_df.drop('county', 'locality')

# fill null count with 1 (minimum presence assumption)
silver_df = silver_df.fillna({'count': 1})

> **Note:** `count` is 98% null in this dataset, filling with 1 represents minimum presence only. For flock size analysis, use eBird `howMany` instead.

#### Uniformity Check
Verifying whether `record_type` and `status` are constant across all records. 

In [ ]:
# check 
silver_df.groupBy('record_type').count().toPandas()

In [ ]:
silver_df.groupBy('status').count().toPandas()

> **Note:** `record_type` and `status` are uniform in the 2025 sample (`HUMAN_OBSERVATION` and `PRESENT` respectively). Both are retained in Silver as future ingestion years may contain other values. Filter in the Gold layer via dbt if needed.

#### Null Audit
Verifying null counts after all transformations before writing to Silver.

In [ ]:
null_counts = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

#### Final Validation
A last look at the data before committing to GCS Silver.

In [ ]:
# convert to pandas for final inspection
df = silver_df.toPandas()
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# distribution of record types
silver_df.groupBy('record_type').count().orderBy('count', ascending=False).toPandas()

In [ ]:
# top 10 states by sighting count
silver_df.groupBy('state').count().orderBy('count', ascending=False).limit(10).toPandas()

In [ ]:
# sightings by year and month
silver_df.groupBy('year', 'month').count().orderBy('year', 'month').toPandas()

#### Write to GCS Silver
Writing the cleaned and transformed dataset to GCS Silver as Parquet, partitioned by year and month for efficient downstream querying.

In [ ]:
SILVER_PATH = f'gs://{GCS_BUCKET}/silver_gbif'

silver_df.write \
    .mode('overwrite') \
    .partitionBy('year', 'month') \
    .parquet(SILVER_PATH)

print(f"Silver data written to: {SILVER_PATH}")
print(f"Final row count: {silver_df.count():,}")